In [4]:
"""
ZTF AGN Lightcurve Pipeline
============================
1. Query SDSS QSO catalog via CasJobs SQL (TOP 1000, deduplicated on bestobjid)
2. For each QSO, fetch g+r lightcurves from IRSA ZTF LC-API (3" cone search)
3. Keep best positional match (closest ZTF oid) per source
4. Filter bad epochs (catflags != 0)
5. Store as HDF5 via pandas HDFStore:
       /ztf_agn_lc/<oid>  →  DataFrame columns: time_g, mag_g, magerr_g,
                                                  time_r, mag_r, magerr_r
   g and r are time-union indexed; NaN where bands don't overlap.

Dependencies:
    pip install astroquery pandas requests numpy
"""

import warnings
from tables import NaturalNameWarning
warnings.filterwarnings("ignore", category=NaturalNameWarning)

import time
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from astroquery.sdss import SDSS
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.io import ascii as astropy_ascii
from io import StringIO

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

OUTPUT_HDF5 = Path("ztf_agn_lc.h5")
HDF5_ROOT   = "ztf_agn_lc"         # top-level HDF5 group

ZTF_LC_API  = "https://irsa.ipac.caltech.edu/cgi-bin/ZTF/nph_light_curves"
SEARCH_RADIUS_DEG = 3.0 / 3600.0   # 3 arcsec in degrees
BAD_CATFLAGS_MASK = 32768           # standard ZTF bad-epoch bitmask
REQUEST_DELAY_SEC = 0.5             # polite delay between IRSA requests
TOT_NUM_SDSS_OBJ = 10

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Step 1 — Query SDSS QSO catalog
# ---------------------------------------------------------------------------

SDSS_SQL = f"""
SELECT TOP {TOT_NUM_SDSS_OBJ}
    s.bestobjid,
    s.ra,
    s.dec,
    s.z        AS redshift,
    s.class,
    s.subclass
FROM SpecObj AS s
WHERE
    s.class    = 'QSO'
    AND s.bestobjid IS NOT NULL
    AND s.bestobjid > 0
GROUP BY
    s.bestobjid, s.ra, s.dec, s.z, s.class, s.subclass
ORDER BY
    s.bestobjid
"""
# GROUP BY bestobjid ensures deduplication: if multiple spectra share the
# same bestobjid (re-observations), only one row survives.


def query_sdss_qsos() -> pd.DataFrame:
    """Run SDSS SQL via astroquery and return a clean DataFrame."""
    log.info("Querying SDSS CasJobs …")
    result = SDSS.query_sql(SDSS_SQL)
    if result is None or len(result) == 0:
        raise RuntimeError("SDSS query returned no results.")
    df = result.to_pandas()
    log.info("SDSS: %d QSOs returned", len(df))

    # Defensive deduplication in pandas (in case SQL GROUP BY left any edge cases)
    before = len(df)
    df = df.drop_duplicates(subset="bestobjid").reset_index(drop=True)
    if len(df) < before:
        log.warning("Dropped %d duplicate bestobjid rows post-SQL", before - len(df))

    return df


# ---------------------------------------------------------------------------
# Step 2 — Fetch ZTF lightcurves from IRSA LC-API
# ---------------------------------------------------------------------------

def fetch_ztf_lc(ra: float, dec: float) -> pd.DataFrame | None:
    params = {
        "POS":               f"CIRCLE {ra} {dec} {SEARCH_RADIUS_DEG}",
        "BANDNAME":          "g,r",
        "BAD_CATFLAGS_MASK": BAD_CATFLAGS_MASK,
        "FORMAT":            "ipac_table",
    }
    try:
        resp = requests.get(ZTF_LC_API, params=params, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as exc:
        log.warning("IRSA request failed for RA=%.5f Dec=%.5f: %s", ra, dec, exc)
        return None

    if not resp.text.strip() or "oid" not in resp.text:
        return None  # empty response

    try:
        table = astropy_ascii.read(resp.text, format="ipac")
    except Exception as exc:
        log.warning("Failed to parse IPAC table for RA=%.5f Dec=%.5f: %s", ra, dec, exc)
        return None

    if len(table) == 0:
        return None

    df = table.to_pandas()

    for col in ["ra", "dec", "hjd", "mag", "magerr", "catflags", "oid"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


# ---------------------------------------------------------------------------
# Step 3 — Select best ZTF match (closest oid per band) and build per-band DataFrames
# ---------------------------------------------------------------------------

def select_best_oid_per_band(
    lc_df: pd.DataFrame, ra: float, dec: float
) -> dict[str, int | None]:
    """
    For each band (zg, zr), pick the oid whose median position is closest
    to the SDSS QSO coordinates. Returns {"zg": oid_or_None, "zr": oid_or_None}.
    """
    source_coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg)
    result = {"zg": None, "zr": None}

    for band in ("zg", "zr"):
        band_df = lc_df[lc_df["filtercode"] == band]
        best_oid, best_sep = None, np.inf
        for oid, grp in band_df.groupby("oid"):
            ztf_coord = SkyCoord(
                ra=grp["ra"].median() * u.deg,
                dec=grp["dec"].median() * u.deg,
            )
            sep = source_coord.separation(ztf_coord).arcsec
            if sep < best_sep:
                best_sep = sep
                best_oid = oid
        result[band] = best_oid

    return result


def build_merged_lightcurve(
    lc_df: pd.DataFrame, oids: dict[str, int | None]
) -> tuple[pd.DataFrame | None, int | None]:
    """
    Build a time-union DataFrame from the best g-band oid and best r-band oid.
    Returns (merged_df, primary_oid) where primary_oid is the r-band oid
    (falling back to g-band if r is absent), used as the HDF5 key.
    """
    frames = []

    oid_g = oids["zg"]
    oid_r = oids["zr"]

    if oid_g is not None:
        g = lc_df[lc_df["oid"] == oid_g][["hjd", "mag", "magerr"]].copy()
        g = g.rename(columns={"hjd": "time_g", "mag": "mag_g", "magerr": "magerr_g"})
        frames.append(g.reset_index(drop=True))

    if oid_r is not None:
        r = lc_df[lc_df["oid"] == oid_r][["hjd", "mag", "magerr"]].copy()
        r = r.rename(columns={"hjd": "time_r", "mag": "mag_r", "magerr": "magerr_r"})
        frames.append(r.reset_index(drop=True))

    if not frames:
        return None, None

    merged = pd.concat(frames, axis=0, ignore_index=True)
    merged["_t"] = merged["time_g"].fillna(merged["time_r"])
    merged = merged.sort_values("_t").drop(columns="_t").reset_index(drop=True)

    primary_oid = oid_r if oid_r is not None else oid_g
    return merged, primary_oid


# ---------------------------------------------------------------------------
# Step 4 — Write to HDF5 via pandas HDFStore
# ---------------------------------------------------------------------------

def save_to_hdf5(oid: int, lc: pd.DataFrame, store: pd.HDFStore) -> None:
    """Write a single object's lightcurve DataFrame into the HDFStore."""
    key = f"/{HDF5_ROOT}/{oid}"
    store.put(key, lc, format="fixed", data_columns=True)


# ---------------------------------------------------------------------------
# Main pipeline
# ---------------------------------------------------------------------------

def main() -> None:
    # --- SDSS ---
    sdss_df = query_sdss_qsos()

    n_total    = len(sdss_df)
    n_matched  = 0
    n_no_match = 0
    n_error    = 0

    with pd.HDFStore(OUTPUT_HDF5, mode="w", complevel=5, complib="blosc") as store:
        for idx, row in sdss_df.iterrows():
            ra, dec = float(row["ra"]), float(row["dec"])
            bestobjid = row["bestobjid"]

            log.info(
                "[%4d/%d] bestobjid=%s  RA=%.5f  Dec=%.5f",
                idx + 1, n_total, bestobjid, ra, dec,
            )

            # --- ZTF cone search ---
            lc_df = fetch_ztf_lc(ra, dec)
            time.sleep(REQUEST_DELAY_SEC)

            if lc_df is None or lc_df.empty:
                log.debug("  → no ZTF detections within cone")
                n_no_match += 1
                continue

            # --- Best positional match per band ---
            oids = select_best_oid_per_band(lc_df, ra, dec)
            if oids["zg"] is None and oids["zr"] is None:
                n_no_match += 1
                continue

            # --- Build merged g+r lightcurve ---
            lc, primary_oid = build_merged_lightcurve(lc_df, oids)
            if lc is None or lc.empty:
                n_error += 1
                continue

            # --- Write to HDF5 (keyed by r-band oid, falling back to g) ---
            key = f"/{HDF5_ROOT}/{primary_oid}"
            if key in store:
                log.warning(
                    "  → oid=%s already in store (duplicate ZTF match); skipping",
                    primary_oid,
                )
                n_error += 1
                continue

            save_to_hdf5(primary_oid, lc, store)
            n_matched += 1
            log.info(
                "  → oid_g=%s  oid_r=%s  g_epochs=%d  r_epochs=%d  saved",
                oids["zg"], oids["zr"],
                lc["mag_g"].notna().sum(),
                lc["mag_r"].notna().sum(),
            )

    # --- Summary ---
    log.info("=" * 60)
    log.info("Pipeline complete.")
    log.info("  SDSS QSOs queried : %d", n_total)
    log.info("  ZTF matches saved : %d  →  %s", n_matched, OUTPUT_HDF5)
    log.info("  No ZTF match      : %d", n_no_match)
    log.info("  Skipped (other)   : %d", n_error)
    log.info("=" * 60)


if __name__ == "__main__":
    main()

16:26:19  INFO      Querying SDSS CasJobs …
16:26:19  INFO      SDSS: 10 QSOs returned
16:26:19  INFO      [   1/10] bestobjid=1197126704423239975  RA=238.23135  Dec=55.93361
16:26:32  INFO        → oid_g=794110100014215  oid_r=794210100017828  g_epochs=1151  r_epochs=1281  saved
16:26:32  INFO      [   2/10] bestobjid=1197126704428810393  RA=238.55945  Dec=43.16305
16:26:47  INFO        → oid_g=721113300015990  oid_r=721213300002028  g_epochs=1010  r_epochs=1129  saved
16:26:47  INFO      [   3/10] bestobjid=1197126704428875838  RA=238.43512  Dec=43.10423
16:27:02  INFO        → oid_g=721113300006600  oid_r=721213300002536  g_epochs=1275  r_epochs=1319  saved
16:27:02  INFO      [   4/10] bestobjid=1197126704428875872  RA=238.49717  Dec=43.15542
16:27:17  INFO        → oid_g=721113300001248  oid_r=721213300002106  g_epochs=1242  r_epochs=1309  saved
16:27:17  INFO      [   5/10] bestobjid=1197126705520378125  RA=239.70157  Dec=2.47184
16:27:33  INFO        → oid_g=481103100000660  oid